# 🚀 AGENT WEB · Colab LLM Worker (GGUF / llama-cpp-python)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

Google Colab の GPU ランタイム（**L4** 推奨）で Google Drive 内の任意の GGUF モデルをロードし、**公式 SSH ポートフォワード**（または Cloudflare トンネル）経由で自宅の AGENT WEB から安全に推論するための実行ノートブックです。

- **公式 SSH 接続（推奨）**: 外部中継を介さず、Google 公式 WebSocket + OpenSSH で完全な閉域・暗号化トンネル直結。
- **接続先 URL**: （常に固定・再入力不要）
- **最新エラー対策完備**: CUDA 12/13 自動適合、 欠損対策、 例外回避、Drive FUSE ハング対策を完全内蔵。


In [ ]:
#@title 1️⃣ 環境自動構築 (CUDA 共有ライブラリ解決 + llama-cpp-python)
#@markdown Colab の実環境（CUDA 12/13）を検出し、適合するビルド済み Wheel を自動導入します。
import ctypes, glob, os, pathlib, subprocess, sys

def setup_llama_cpp():
    cuda_tag = "cu124"
    try:
        import torch
        if torch.cuda.is_available() and torch.version.cuda:
            ver = torch.version.cuda.replace(".", "")[:3]
            if ver in ("121", "122", "123", "124", "125", "130", "132"):
                cuda_tag = f"cu{ver}"
            elif ver.startswith("13"):
                cuda_tag = "cu130"
            elif ver.startswith("12"):
                cuda_tag = "cu125"
    except Exception:
        pass

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12", "gradio"
    ])

    sys_lib_dir = pathlib.Path("/usr/lib/x86_64-linux-gnu")
    search_roots = [p for p in sys.path if "site-packages" in p or "dist-packages" in p]
    so_patterns = ["libcudart.so*", "libcublasLt.so*", "libcublas.so*"]
    found_libs = []
    for root in search_roots:
        for pat in so_patterns:
            for so_path in glob.glob(os.path.join(root, "nvidia", "*", "lib", pat)):
                found_libs.append(so_path)
                if sys_lib_dir.exists():
                    dst = sys_lib_dir / os.path.basename(so_path)
                    if not dst.exists():
                        try: os.symlink(so_path, dst)
                        except Exception: pass
    try: subprocess.run(["ldconfig"], check=False)
    except Exception: pass
    for so_path in sorted(set(found_libs)):
        try: ctypes.CDLL(so_path, mode=ctypes.RTLD_GLOBAL)
        except Exception: pass

    candidates = [cuda_tag, "cu124", "cu130", "cu125"]
    seen = []
    for tag in candidates:
        if tag in seen: continue
        seen.append(tag)
        index_url = f"https://abetlen.github.io/llama-cpp-python/whl/{tag}"
        print(f"Installing llama-cpp-python ({tag})...")
        try:
            subprocess.check_call([
                sys.executable, "-m", "pip", "install", "-q",
                "--upgrade", "--force-reinstall", "--only-binary=:all:",
                "llama-cpp-python", "--extra-index-url", index_url
            ])
            subprocess.check_call([sys.executable, "-c", "from llama_cpp import Llama; print('llama-cpp-python verified successfully')"])
            print("✅ インストール完了")
            return
        except Exception as e:
            print(f"⚠️ {tag} での導入に失敗、フォールバックを試行します: {e}")
    raise RuntimeError("llama-cpp-python のインストールに失敗しました。")

setup_llama_cpp()


In [ ]:
#@title 2️⃣ Google Drive マウント & モデル選択・ロード
#@markdown フォームから対象モデル名やコンテキスト長を設定できます。
import ctypes, glob, os, pathlib, sys
from google.colab import drive

drive.mount("/content/drive")

MODEL_NAME = "Huihui-Qwen3.8-27B-abliterated-GSQ-RCO-IQ3_S.gguf" #@param {type:"string"}
MODEL_DIR = "/content/drive/MyDrive/colab-model" #@param {type:"string"}
CONTEXT_SIZE = 8192 #@param [2048, 4096, 8192, 16384, 32768] {type:"raw"}
N_GPU_LAYERS = -1 #@param {type:"integer"}

model_dir_path = pathlib.Path(MODEL_DIR)
model_path = model_dir_path / MODEL_NAME
if not model_path.exists():
    if model_dir_path.exists():
        candidates = sorted(model_dir_path.glob("*.gguf"))
        if candidates:
            print(f"指定モデルが見つかりません。{MODEL_DIR} 内の検出モデルを使用します: {candidates[0].name}")
            model_path = candidates[0]
            MODEL_NAME = model_path.name
    if not model_path.exists():
        all_found = list(pathlib.Path("/content/drive/MyDrive").rglob(MODEL_NAME))
        if all_found:
            model_path = all_found[0]

assert model_path.exists(), f"モデルファイルが見つかりません: {model_path}"
MODEL_PATH = str(model_path)
size_gb = os.path.getsize(MODEL_PATH) / (1024 ** 3)
print(f"対象モデル: {MODEL_PATH} ({size_gb:.2f} GB)")

for _r in sys.path:
    if "packages" in _r:
        for _so in sorted(glob.glob(_r + "/nvidia/*/lib/libcu*.so*")):
            try: ctypes.CDLL(_so, mode=ctypes.RTLD_GLOBAL)
            except Exception: pass

for _name, _fd in (("stdout", 1), ("stderr", 2)):
    _stream = getattr(sys, _name, None)
    if _stream is not None:
        try: _stream.fileno()
        except Exception:
            try: _stream.fileno = (lambda fd=_fd: fd)
            except Exception: pass

from llama_cpp import Llama

print(f"GPU (VRAM) へ展開中... (コンテキスト長: {CONTEXT_SIZE})")
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=CONTEXT_SIZE,
    n_gpu_layers=N_GPU_LAYERS,
    n_threads=8,
    use_mmap=False,
    verbose=True
)
print("✅ LLM ロード完了:", MODEL_NAME)


In [ ]:
#@title 3️⃣ 単体動作テスト（省略可能）
TEST_PROMPT = "こんにちは！自己紹介を1文でお願いします。" #@param {type:"string"}
res = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "あなたは有能なAIアシスタントです。"},
        {"role": "user", "content": TEST_PROMPT}
    ],
    max_tokens=256,
    temperature=0.7
)
print("
--- 応答 ---")
print(res["choices"][0]["message"]["content"].strip())


In [ ]:
#@title 4️⃣ AGENT WEB 連携 Worker 起動
import base64, hmac, hashlib, io, json, os, re, secrets, subprocess, threading, time, uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

def start_worker(llm=None, llm_model_name=None, port=8765):
    import torch
    token = secrets.token_urlsafe(32)
    jobs = {}
    lock = threading.Lock()
    instance_id = str(uuid.uuid4())

    class Handler(BaseHTTPRequestHandler):
        def log_message(self, *args): pass
        def respond(self, status, body):
            raw = json.dumps(body).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(raw)))
            self.end_headers()
            self.wfile.write(raw)
        def authorized(self):
            if not hmac.compare_digest(self.headers.get("Authorization", ""), "Bearer " + token):
                self.respond(401, {"error": "Unauthorized"})
                return False
            return True
        def do_GET(self):
            if not self.authorized(): return
            if self.path == "/status":
                with lock: busy = any(j.get("status") == "running" for j in jobs.values())
                self.respond(200, {
                    "gpu": torch.cuda.get_device_name(0),
                    "model": llm_model_name or "GGUF LLM",
                    "busy": busy,
                    "protocol": 1,
                    "instance_id": instance_id,
                    "capabilities": {"reference_image": False, "text_to_image": False, "motion": False, "text_generation": llm is not None}
                })
            elif self.path.startswith("/jobs/"):
                job_id = self.path.split("/")[-1]
                with lock: job = dict(jobs.get(job_id, {}))
                if not job: return self.respond(404, {"error": "Job not found"})
                self.respond(200, job)
            else: self.respond(404, {"error": "Not found"})
        def do_POST(self):
            if not self.authorized(): return
            if self.path == "/chat":
                if llm is None: return self.respond(400, {"error": "LLM is not loaded"})
                try:
                    length = int(self.headers.get("Content-Length", 0))
                    body = json.loads(self.rfile.read(length))
                    prompt = body.get("prompt", "")
                    system_prompt = body.get("system_prompt", "あなたは有能なAIアシスタントです。")
                    max_tokens = int(body.get("max_tokens", 512))
                    temperature = float(body.get("temperature", 0.7))
                    job_id = body.get("id") or str(uuid.uuid4())
                    with lock:
                        if any(j.get("status") == "running" for j in jobs.values()):
                            return self.respond(409, {"error": "GPU is busy"})
                        jobs[job_id] = {"id": job_id, "status": "running", "progress": 0}
                    try:
                        res = llm.create_chat_completion(
                            messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}],
                            max_tokens=max_tokens,
                            temperature=temperature,
                        )
                        text = res["choices"][0]["message"]["content"]
                        usage = res.get("usage", {})
                        with lock: jobs[job_id].update(status="completed", progress=100, text=text)
                        return self.respond(200, {"id": job_id, "status": "completed", "text": text, "model": llm_model_name or "GGUF LLM", "usage": usage})
                    except Exception as exc:
                        with lock: jobs[job_id].update(status="failed", error=str(exc)[:2000])
                        return self.respond(400, {"error": str(exc)[:2000]})
                except Exception as exc:
                    return self.respond(400, {"error": str(exc)})
            self.respond(404, {"error": "Not found"})

    server = ThreadingHTTPServer(("127.0.0.1", port), Handler)
    threading.Thread(target=server.serve_forever, daemon=True).start()
    return server, token

if "aw_server" in globals():
    try: aw_server.shutdown(); aw_server.server_close()
    except Exception: pass

aw_server, aw_token = start_worker(llm=llm, llm_model_name=MODEL_NAME, port=8765)
Path("/content/token.txt").write_text(aw_token)

print("=" * 64)
print("【1. Google 公式 SSH 接続（推奨）】")
print("VM 側でポートフォワードを実行（すでに実行中の場合は不要です）:")
print("  ssh -N -f -L 8765:localhost:8765 colab
")
print("AGENT WEB の登録情報:")
print("  接続URL : http://127.0.0.1:8765")
print(f"  接続キー: {aw_token}")
print("=" * 64)

# Cloudflare トンネルのフォールバック起動
try:
    bin_path = Path("/content/cloudflared")
    if not bin_path.exists():
        import urllib.request
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", bin_path)
        bin_path.chmod(0o700)
    if bin_path.exists():
        log = open("/content/tunnel.log", "w+")
        aw_tunnel = subprocess.Popen([str(bin_path), "tunnel", "--url", "http://127.0.0.1:8765", "--no-autoupdate"], stdout=log, stderr=subprocess.STDOUT)
        time.sleep(3)
        log.seek(0)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", log.read())
        if m:
            print("
【2. Cloudflare トンネル (予備フォールバック)】")
            print(f"  接続URL: {m.group(0)}")
            print(f"  接続キー: {aw_token}")
except Exception: pass


In [ ]:
#@title 5️⃣ 停止・VRAM解放
if "aw_server" in globals():
    aw_server.shutdown(); aw_server.server_close()
if "aw_tunnel" in globals():
    aw_tunnel.terminate()
if "llm" in globals():
    del llm
import gc, torch
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Worker を停止し、VRAM を解放しました。")
